# CloudVision — GitHub ↔ Kaggle GPU worker

شغّل هذه الخلية داخل جلسة Kaggle الحالية بعد اختيار GPU T4. هذا العامل يستقبل مهام تحسين الفيديو من GitHub، ويعالج الإطارات تباعاً باستخدام Real-ESRGAN وFFmpeg، ثم يحفظ ملف MP4 في /kaggle/working/cloudvision ويرفع تقرير التنفيذ إلى GitHub.

يمكن إرسال رابط فيديو YouTube مسجّل مباشرة ضمن المهمة؛ ينزّله العامل إلى Kaggle تلقائياً. لا يلزم رفع ملف محلي. البث الحي غير مدعوم في هذه النسخة. لا تضع أي مفتاح داخل نص الدفتر.

In [ ]:
import os, sys, subprocess, requests
from kaggle_secrets import UserSecretsClient

# خزّن GitHub fine-grained PAT في Kaggle Secrets باسم GITHUB_TOKEN.
# يجب أن يملك صلاحية Contents: Read and write لهذا المستودع فقط.
token = UserSecretsClient().get_secret('GITHUB_TOKEN')
if not token:
    raise RuntimeError('أضف GITHUB_TOKEN إلى Kaggle Secrets أولاً؛ لا ترسله في المحادثة.')
os.environ['GITHUB_TOKEN'] = token
os.environ['CLOUDVISION_ONCE'] = '1'
os.environ['CLOUDVISION_WAIT_SECONDS'] = '900'

url = 'https://raw.githubusercontent.com/khd98me-svg/TermuxAgent/a5582c24aa78a1bdc8ed6b80974b3dd6c44869f6/cloudvision/kaggle_bridge.py'
response = requests.get(url, timeout=30)
response.raise_for_status()
script_path = '/kaggle/working/kaggle_bridge.py'
with open(script_path, 'w', encoding='utf-8') as f:
    f.write(response.text)
print('Bridge script downloaded. Starting listener...')
print('Keep this cell running; interrupt it to stop the bridge.')
subprocess.run([sys.executable, script_path], check=True)

## التشغيل والأمان
- أضف GitHub fine-grained token بصلاحية Contents Read/Write للمستودع المحدد إلى Kaggle Secrets بالاسم `GITHUB_TOKEN`.
- لا ترسل Token أو Cookies أو Session IDs في المحادثة.
- يدعم العامل ثلاثة إجراءات ثابتة فقط: `cuda_info` و`realesrgan_image_test` و`video_upscale`.
- مهمة الفيديو تقبل رابط YouTube مسجّلاً عبر `source_url` أو مساراً محلياً داخل `/kaggle/input` أو `/kaggle/working`، وتحفظ الناتج في `/kaggle/working/cloudvision/` مع محاولة الحفاظ على الصوت الأصلي. البث الحي غير مدعوم بعد.
- اترك الخلية تعمل أثناء تنفيذ المهمة. Kaggle ليس خادماً دائماً وقد ينهي الجلسة وفق حصصه وسياساته.
- بعد اكتمال المهمة، يظهر تقرير JSON في `cloudvision/bridge-results/` على GitHub، ويكون ملف MP4 ضمن ملفات/مخرجات جلسة Kaggle.